# Data Cleaning and EDA

This notebook is used for the data cleaning for crm_sales, segmenation and targeting.
It additionally explores the CRM dataset to impove data quality and allow for further analysis. 


# Cleaning and EDA

## 1. Accounts
crm_sales.crm_raw.accounts

In [0]:
%sql
SELECT 
   *
FROM 
    crm_sales.crm_raw.accounts

In [0]:
%sql
--checking for null values
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN account IS NULL OR TRIM(account) = '' 
        THEN 1 ELSE 0 END) AS account_missing,

    SUM(CASE WHEN sector IS NULL OR TRIM(sector) = '' 
        THEN 1 ELSE 0 END) AS sector_missing,

    SUM(CASE WHEN year_established IS NULL OR TRIM(CAST(year_established AS STRING)) = ''
        THEN 1 ELSE 0 END) AS year_established_missing,

    SUM(CASE 
        WHEN revenue IS NULL OR TRIM(CAST(revenue AS STRING)) = ''
        THEN 1 ELSE 0 END) AS revenue_missing,

    SUM(CASE WHEN employees IS NULL OR TRIM(CAST(employees AS STRING)) = ''
        THEN 1 ELSE 0 END) AS employees_missing,

    SUM(CASE WHEN office_location IS NULL OR TRIM(office_location) = ''
        THEN 1 ELSE 0 END) AS office_location_missing,

    SUM(CASE 
        WHEN subsidiary_of IS NULL OR TRIM(subsidiary_of) = ''
        THEN 1 ELSE 0 END) AS subsidiary_of_missing

FROM crm_sales.crm_raw.accounts;


In [0]:
%sql
SELECT 
DISTINCT account
FROM 
crm_sales.crm_raw.accounts


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS crm_sales.crm_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE crm_sales.crm_clean.accounts AS

SELECT
    CONCAT('act_', ROW_NUMBER() OVER (ORDER BY account) + 100) AS account_id,
    TRIM(account) AS account,
    TRIM(sector) AS sector,
    TRY_CAST(NULLIF(TRIM(year_established), '') AS date) AS year_established,
    TRY_CAST(NULLIF(TRIM(revenue), '') AS DECIMAL(15,2)) AS revenue,
    TRY_CAST(NULLIF(TRIM(employees), '') AS INT) AS employees,
    TRIM(office_location) AS office_location,
    COALESCE(NULLIF(TRIM(subsidiary_of), ''), 'Independent') AS subsidiary_of

FROM crm_sales.crm_raw.accounts;

SELECT 
    *
FROM 
    crm_sales.crm_clean.accounts

# 2. Products

Cleaning and creating a new schema for the products table

In [0]:
%sql
SELECT 
   *
FROM 
    crm_sales.crm_raw.products

In [0]:
%sql
--checking for null values
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN product IS NULL OR TRIM(product) = '' 
        THEN 1 ELSE 0 END) AS product_missing,

    SUM(CASE WHEN series IS NULL OR TRIM(series) = '' 
        THEN 1 ELSE 0 END) AS series_missing,

    SUM(CASE WHEN sales_price IS NULL OR TRIM(CAST(sales_price AS STRING)) = ''
        THEN 1 ELSE 0 END) AS sales_price_missing
FROM 
    crm_sales.crm_raw.products;


In [0]:
%sql
CREATE OR REPLACE TABLE crm_sales.crm_clean.products AS

SELECT
    CONCAT('prd_', ROW_NUMBER() OVER (ORDER BY product) + 00) AS product_id,
    TRIM(product) AS product,
    TRIM(series) AS series,
    TRY_CAST(NULLIF(TRIM(sales_price), '') AS INT) AS sales_price

FROM crm_sales.crm_raw.products;

SELECT 
    *
FROM 
    crm_sales.crm_clean.products

# Sales

Cleaning and creating a new schema for the sales table

In [0]:
%sql
SELECT 
   *
FROM 
    crm_sales.crm_raw.sales_teams

In [0]:
%sql
--checking for null values
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN sales_agent IS NULL OR TRIM(sales_agent) = '' 
        THEN 1 ELSE 0 END) AS sales_agent_missing,

    SUM(CASE WHEN manager IS NULL OR TRIM(manager) = '' 
        THEN 1 ELSE 0 END) AS manager_missing,

    SUM(CASE WHEN regional_office IS NULL OR TRIM(CAST(regional_office AS STRING)) = ''
        THEN 1 ELSE 0 END) AS sales_price_missing
FROM 
    crm_sales.crm_raw.sales_teams;

In [0]:
%sql
CREATE OR REPLACE TABLE crm_sales.crm_clean.sales_teams AS

SELECT
    CONCAT('rep_', ROW_NUMBER() OVER (ORDER BY sales_agent) + 00) AS sales_agent_id,
    TRIM(sales_agent) AS sales_agent,
    TRIM(manager) AS manager,
    TRIM(regional_office) AS regional_office

FROM crm_sales.crm_raw.sales_teams;

SELECT 
    *
FROM 
    crm_sales.crm_clean.sales_teams

### Sales Pipeline
Cleaning and creating a schema for the sales_pipeline table.

In [0]:
%sql
SELECT 
   *
FROM 
    crm_sales.crm_raw.sales_pipeline

In [0]:
%sql
--checking for null values
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN opportunity_id IS NULL OR TRIM(opportunity_id) = '' 
        THEN 1 ELSE 0 END) AS opportunity_id_missing, 

    SUM(CASE WHEN sales_agent IS NULL OR TRIM(sales_agent) = '' 
        THEN 1 ELSE 0 END) AS sales_agent_missing,

    SUM(CASE WHEN product IS NULL OR TRIM(product) = '' 
        THEN 1 ELSE 0 END) AS product_missing,

    SUM(CASE WHEN account IS NULL OR TRIM(account) = '' 
        THEN 1 ELSE 0 END) AS account_missing,

    SUM(CASE WHEN deal_stage IS NULL OR TRIM(deal_stage) = '' 
        THEN 1 ELSE 0 END) AS deal_stage_missing,

    SUM(CASE WHEN engage_date IS NULL OR TRIM(engage_date) = '' 
        THEN 1 ELSE 0 END) AS engage_date_missing,

    SUM(CASE WHEN close_date IS NULL OR TRIM(close_date) = '' 
        THEN 1 ELSE 0 END) AS close_date_missing,

    SUM(CASE WHEN close_value IS NULL OR TRIM(CAST(close_value AS STRING)) = '' 
        THEN 1 ELSE 0 END) AS close_value_missing

FROM 
    crm_sales.crm_raw.sales_pipeline;

In [0]:
%sql
-- All nulls are from engaging or prospecting deal stage.
SELECT 
    *
FROM 
    crm_sales.crm_raw.sales_pipeline
WHERE 
    engage_date IS NULL OR close_date IS NULL OR close_value IS NULL

In [0]:
%sql
CREATE OR REPLACE TABLE crm_sales.crm_clean.sales_pipeline AS 

SELECT
    TRIM(opportunity_id) AS opportunity_id,
    TRIM(sales_agent) AS sales_agent,
    TRIM(product) AS product,
    TRIM(account) AS account,
    COALESCE(account, 'Unknown Account') AS account_name_clean,
    TRIM(deal_stage) AS deal_stage,
    TRY_CAST(NULLIF(TRIM(engage_date), '') AS date) AS engage_date,
    TRY_CAST(NULLIF(TRIM(close_date), '') AS date) AS close_date,
    TRY_CAST(NULLIF(TRIM(close_value), '') AS INT) AS close_value

FROM crm_sales.crm_raw.sales_pipeline;

SELECT 
    *
FROM 
    crm_sales.crm_clean.sales_pipeline

## Validating primary keys

In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    COUNT(account_id) AS non_null_ids,
    COUNT(DISTINCT account_id) AS unique_ids

FROM 
 crm_sales.crm_clean.accounts;

 SELECT 
    COUNT(*) AS total_rows,
    COUNT(product_id) AS non_null_ids,
    COUNT(DISTINCT product_id) AS unique_ids

FROM 
 crm_sales.crm_clean.products;

SELECT 
    COUNT(*) AS total_rows,
    COUNT(sales_agent_id) AS non_null_ids,
    COUNT(DISTINCT sales_agent_id) AS unique_ids

FROM 
 crm_sales.crm_clean.sales_teams;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_sales_pipeline_relational AS

SELECT 
    sales_pipeline.*,
    accounts.account_id,
    products.product_id,
    sales_teams.sales_agent_id,
    sales_teams.manager,
    sales_teams.regional_office
FROM 
    crm_sales.crm_clean.sales_pipeline AS sales_pipeline
LEFT JOIN 
    crm_sales.crm_clean.products AS products 
    ON sales_pipeline.product = products.product
LEFT JOIN 
    crm_sales.crm_clean.sales_teams AS sales_teams 
    ON sales_pipeline.sales_agent = sales_teams.sales_agent
LEFT JOIN 
    crm_sales.crm_clean.accounts AS accounts 
    ON sales_pipeline.account = accounts.account;

SELECT
    *
FROM 
    vw_sales_pipeline_relational


## Validation and Testing

In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT account_id) AS unique_ids,
    SUM(CASE WHEN account_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM
    crm_sales.crm_clean.accounts


In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT account_id) AS unique_ids,
    SUM(CASE WHEN account_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM
    crm_sales.crm_clean.accounts